In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## **0. Definition**

In [2]:
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/Legal_Sytem/01_Dataset/raw")
PROCESSED_DIR = Path("/content/drive/MyDrive/Legal_Sytem/01_Dataset/02_processed_data")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

DATA_FILES = {
    'train': DATA_DIR / 'train.xlsx',
    'val': DATA_DIR / 'val.xlsx',
    'test': DATA_DIR / 'test.xlsx'
}

COLUMNS= [
    "index",
    "context",
    "article",
    "document",
    "question",
    "extractive answer",
    "abstractive answer",
    "yes/no"
]

NORMALIZED_COLUMNS= [
    "context",
    "article",
    "document",
    "question",
    "extractive answer",
    "abstractive answer"
]

REQUIED_NON_EMPTY_COLUMNS = [
    "context",
    "article",
    "document",
    "question"
]

## **01. Normalize data**

In [3]:
import pandas as pd
import numpy as np
import json
import unicodedata
import re

def normalize_text(text: str)-> str:
  if pd.isna(text):
    return ""
  text = unicodedata.normalize("NFKC", str(text))

  text = text.replace("\u00a0", " ")

  text = text.replace("\r\n", "\n").replace("\r", "\n")

  text = re.sub(r"[ \t]+", " ", text)

  text = re.sub(r" *\n *", "\n", text)

  text = re.sub(r"\n{3,}", "\n\n", text)

  return text.strip()

In [4]:
def normalize_for_comparison(text: str)-> str:
  text = normalize_text(text).lower()
  text = re.sub(r"\s+", " ", text)
  return text.strip()

In [5]:
# Tạo index chuẩn các các trường thông tin
import hashlib

def stable_id(prefix: str, *values: str, length: int=20) -> str:
  payload = "|||".join(normalize_for_comparison(v) for v in values)

  digest = hashlib.sha256(payload.encode("utf-8")).hexdigest()[:length]

  return f"{prefix}_{digest}"

In [6]:
# Chuẩn hóa giá trị để khi lưu tránh bị lỗi.
import numpy as np

def json_safe(value):
  if pd.isna(value):
    return None # Chuan hoa ve None de khi luu dang json thi tu dong chuyen thanh null ma khong loi

  if isinstance(value, np.generic):
    return value.item()

  return value

In [7]:
# Hàm lưu file dạng jsonl
def dataframe_to_jsonl(df: pd.DataFrame, output_path: Path) -> None:
  with output_path.open("w", encoding= "utf-8") as file:

    for row in df.to_dict(orient= "records"):
      record = {key: json_safe(value) for key, value in row.items()}

      file.write(json.dumps(record, ensure_ascii= False) + "\n")

In [8]:
def save_json(data, path: Path) -> None:
  with path.open("w", encoding= "utf-8") as file:
    json.dump(data, file, ensure_ascii= False, indent= 2)

## **02. Statistics**

In [9]:
# Thong ke so dong du lieu cua cac file goc

raw_splits = {}

for split_name, file_path in DATA_FILES.items():
  if not file_path.exists():
    raise FileExistsError(
        f"Không tìm thấy file: {file_path}.\nVui lòng kiểm tra lại đường dẫn."
    )

  dataframe = pd.read_excel(file_path)

  missing_columns = [
      column for column in COLUMNS if column not in dataframe.columns
  ]

  if missing_columns:
    raise ValueError(
        f"Tệp '{set}' thiếu cột: {missing_columns}.\nCột hiện có: {dataframe.columns}"
    )

  dataframe = dataframe[COLUMNS].copy()
  raw_splits[split_name] = dataframe

  print(f"{split_name}: {len(dataframe):,} row & {len(dataframe.columns):,} columns")


print(f"Tổng số dòng raw: {sum(len(df) for df in raw_splits.values()):,}")

train: 7,806 row & 8 columns
val: 976 row & 8 columns
test: 976 row & 8 columns
Tổng số dòng raw: 9,758


In [10]:
for split_name, dataframe in raw_splits.items():
  invalid_mask = pd.Series(False, index=dataframe.index)

  for col in REQUIED_NON_EMPTY_COLUMNS:
    invalid_mask |= dataframe[col].eq("")

  invalid_count = int(invalid_mask.sum())

  if invalid_count:
    print(f"{split_name}: loại {invalid_count} dòng thiếu trường bắt buộc")
    raw_splits[split_name] = dataframe[~invalid_mask].reset_index(drop= True)
  else:
    print(f"{split_name}: không có dòng thiếu trường bắt buộc")

train: không có dòng thiếu trường bắt buộc
val: không có dòng thiếu trường bắt buộc
test: không có dòng thiếu trường bắt buộc


### **2.1. Khôi phục giá trị cột document**

In [12]:
REFERENCE_SPLIT = 'train'

[{REFERENCE_SPLIT}]

[{'train'}]

In [14]:
processed_splits = {}

for split_name, dataframe in raw_splits.items():
  dataframe['context_key'] = dataframe['context'].map(normalize_for_comparison)

reference_dataframe = raw_splits['train']

nunique_context = reference_dataframe['context_key'].nunique()
unique_location = reference_dataframe.drop_duplicates(['document', 'article']).shape[0]

context_to_many_location = (reference_dataframe.drop_duplicates(['context_key', 'document', 'article']).groupby('context_key').size().gt(1).sum())

print(f"[{'train'}] unique context      : {nunique_context:,}")
print(f"[{'train'}] unique location (document, article)    : {unique_location:,}")
print(f"[{'train'}] context has more than 1 location: {context_to_many_location:,}")

assert context_to_many_location == 0 , ("Split tham chiếu không phải ánh xạ 1-1, không thể dùng để khôi phục document.")

[train] unique context      : 294
[train] unique location (document, article)    : 294
[train] context has more than 1 location: 0


In [16]:
context_to_document = reference_dataframe.drop_duplicates('context_key').set_index('context_key')['document'].to_dict()

context_to_article = reference_dataframe.drop_duplicates('context_key').set_index('context_key')['article'].to_dict()

repair_report = []

for split_name, dataframe in raw_splits.items():
  recovered_document = dataframe['context_key'].map(context_to_document)
  recovered_article = dataframe['context_key'].map(context_to_article)

  unmapped_count = int(recovered_document.isna().sum())
  document_mismatch = int((dataframe['document'] != recovered_document).fillna(False).sum())
  article_mismatch = int((dataframe['article'] != recovered_article).fillna(False).sum())

  dataframe['document_raw'] = dataframe['document']
  dataframe['article_raw'] = dataframe['article']

  dataframe['document'] = recovered_document.fillna(dataframe['document'])
  dataframe['article'] = recovered_article.fillna(dataframe['article'])

  repair_report.append({
      'split': split_name,
      'number_of_rows': len(dataframe),
      'document_repaired': document_mismatch,
      'article_repaired': article_mismatch,
      'context_not_in_reference': unmapped_count,
  })

repair_report_df = pd.DataFrame(repair_report)

repair_report_df = repair_report_df.set_index('split')

display(repair_report_df)

,number_of_rows,document_repaired,article_repaired,context_not_in_reference
split,,,,
train,7806,0,0,0
val,976,0,0,0
test,976,907,0,0


In [17]:
# Xác nhận lại: sau khi sửa, mọi context chỉ còn ứng với đúng 1 (document, article).


combined_dataframe = pd.concat(raw_splits.values(), ignore_index= True)

location_per_context = (combined_dataframe.drop_duplicates(['context_key', 'document', 'article']).groupby('context_key').size())

print(f"Tổng số context duy nhất       : {combined_dataframe['context_key'].nunique():,}")
print(f"Context ứng với >1 (doc, art)  : {int(location_per_context.gt(1).sum()):,}")

assert location_per_context.max() == 1, "Vẫn còn context bị gán nhiều vị trí khác nhau."

Tổng số context duy nhất       : 294
Context ứng với >1 (doc, art)  : 0


### **2.2. Create stable ID**

In [21]:
for split_name, dataframe in raw_splits.items():
  dataframe = dataframe.copy()

  dataframe['doc_id'] = [stable_id("DOC", document, article) for document, article in zip(dataframe['document'], dataframe['article'])]

  dataframe['content_hash'] = dataframe['context'].map(lambda context: stable_id("CONTENT", context))

  dataframe['query'] = dataframe['question']

  dataframe['query_id'] = dataframe['question'].map(lambda question: stable_id("QUERY", question))

  dataframe['qa_id'] = [stable_id("QA", query, document, article) for query, document, article in zip(dataframe['question'], dataframe['document'], dataframe['article'])]

  dataframe['split'] = split_name

  dataframe['source_index'] = dataframe['index']

  raw_splits[split_name] = dataframe

print("Đã sinh doc_id / content_hash / query_id / qa_id.")

Đã sinh doc_id / content_hash / query_id / qa_id.


In [26]:
# doc_id và content_hash phải là song ánh. Nếu lệch nghĩa là còn dữ liệu bẩn chưa xử lý.

combined_dataframe = pd.concat(raw_splits.values(), ignore_index=True)

doc_id_per_content = combined_dataframe.groupby('content_hash')['doc_id'].nunique()
content_per_doc_id = combined_dataframe.groupby('doc_id')['content_hash'].nunique()

print(f"Số doc_id duy nhất       : {combined_dataframe['doc_id'].nunique():,}")
print(f"Số content_hash duy nhất : {combined_dataframe['content_hash'].nunique():,}")
print(f"content_hash nhiều doc_id : {int(doc_id_per_content.gt(1).sum()):,}")
print(f"doc_id nhiều content_hash : {int(content_per_doc_id.gt(1).sum()):,}")

assert doc_id_per_content.max() == 1 and content_per_doc_id.max() == 1, (
    "doc_id và content_hash không song ánh."
)

Số doc_id duy nhất       : 294
Số content_hash duy nhất : 294
content_hash nhiều doc_id : 0
doc_id nhiều content_hash : 0


### **2.3. Drop duplicate**

In [27]:
processed_splits = {}
DROP_DUPLICATE_QA_WITHIN_SPLIT = True

for split_name, dataframe in raw_splits.items():
  dataframe = dataframe.copy()

  before_duplicate_len = len(dataframe)

  # Lọc các mẫu trùng cả câu hỏi lẫn điều luật
  if DROP_DUPLICATE_QA_WITHIN_SPLIT:
    dataframe = dataframe.drop_duplicates(subset=['qa_id'], keep='first').copy()

  removed_duplicate_records = before_duplicate_len - len(dataframe)

  dataframe = dataframe.reset_index(drop=True)
  processed_splits[split_name] = dataframe

  print(f"{split_name}: {before_duplicate_len:,} --> {len(dataframe):,} (duplicate QA: {removed_duplicate_records:,})")

train: 7,806 --> 7,718 (duplicate QA: 88)
val: 976 --> 971 (duplicate QA: 5)
test: 976 --> 976 (duplicate QA: 0)


### **2.2. Data Sumarization**

In [28]:
sumary = []

for split_name, df in processed_splits.items():
  sumary.append({
      'split': split_name,
      'number_of_rows': len(df),
      'number_of_columns': len(df.columns),
      'number_of_unique_documents': df['doc_id'].nunique(),
      'number_of_unique_queries': df['query_id'].nunique(),
      'number_of_unique_qa_pairs': df['qa_id'].nunique(),
      'extractive_answer': df['extractive answer'].ne("").sum(),
      'abstractive_answer': df['abstractive answer'].ne("").sum(),
      'yes_no': df['yes/no'].ne("").sum(),
      'avg_question_chars': round(df['question'].str.len().mean(), 2),
      'avg_context_chars': round(df['context'].str.len().mean(), 2),
      'max_question_chars': df['question'].str.len().max(),
      'max_context_chars': df['context'].str.len().max(),
  })

sumary_df = pd.DataFrame(sumary)

sumary_df = sumary_df.set_index('split')

for column in sumary_df.columns:
  sumary_df[column] = sumary_df[column].astype(str)

display(sumary_df.T)

split,train,val,test
number_of_rows,7718,971,976
number_of_columns,18,18,18
number_of_unique_documents,294,258,256
number_of_unique_queries,7697,971,976
number_of_unique_qa_pairs,7718,971,976
extractive_answer,7718,971,976
abstractive_answer,7718,971,976
yes_no,7718,971,976
avg_question_chars,77.05,76.8,77.68
avg_context_chars,1707.63,1678.58,1728.86


In [29]:
sumary_df.to_csv(PROCESSED_DIR / "dataset_sumary.csv", index= False, encoding= 'utf-8-sig')

### **2.3. Thống kê về tỷ lệ overlap giữa các tập dữ liệu**

In [30]:
# Thong ke overlap giua tung tap du lieu

def overlap_report(left_name: str, right_name: str) -> dict:
  left = processed_splits[left_name]
  rigth = processed_splits[right_name]

  left_question = set(left['question'].map(normalize_for_comparison))
  right_question = set(rigth['question'].map(normalize_for_comparison))

  left_qa_ids = set(left['qa_id'])
  right_qa_ids = set(rigth['qa_id'])

  left_passage_ids = set(left['doc_id'])
  right_passage_ids = set(rigth['doc_id'])

  left_query_ids = set(left['query_id'])
  right_query_ids = set(rigth['query_id'])

  report = {
      'left_split': left_name,
      'right_split': right_name,
      'exact_question_overlap': len(left_question & right_question),
      'exact_qa_id_overlap': len(left_qa_ids & right_qa_ids),
      'exact_doc_id_overlap': len(left_passage_ids & right_passage_ids),
      'exact_query_id_overlap': len(left_query_ids & right_query_ids)
  }

  return report

In [31]:
overlap_df = pd.DataFrame([overlap_report('train', 'val'), overlap_report('val', 'test'), overlap_report('train', 'test')])

for col in overlap_df.columns:
  overlap_df[col] = overlap_df[col].astype(str)

display(overlap_df)

,left_split,right_split,exact_question_overlap,exact_qa_id_overlap,exact_doc_id_overlap,exact_query_id_overlap
0,train,val,38,33,258,38
1,val,test,5,4,232,5
2,train,test,15,15,256,15


In [32]:
overlap_df.to_csv(PROCESSED_DIR / 'overlap_report.csv', index= False, encoding= 'utf-8-sig')

- `exact_question_overlap`: cùng một câu hỏi xuất hiện ở 2 tập.
- `exact_qa_id_overlap`: cùng câu hỏi và passage xuất hiện ở 2 tập.
- `doc_id_overlap`: cùng passage xuất hiện ở 2 tập.

Có thể trùng `doc_id`, nhưng không thể trùng câu hỏi `qa_id`.

### **2.4. Lọc trùng dữ liệu các tập val, test so với train.**

In [33]:
CREATE_STRICT_EVAL_SPLITS = True

if CREATE_STRICT_EVAL_SPLITS:
  train_query_ids = set(processed_splits['train']['query_id'])

  strict_val = processed_splits['val'].loc[
      ~processed_splits['val']['query_id'].isin(train_query_ids)
  ].copy()

  train_and_val_query_ids = train_query_ids | set(processed_splits['val']['query_id'])

  strict_test = processed_splits['test'].loc[
      ~processed_splits['test']['query_id'].isin(train_and_val_query_ids)
  ].copy()

  processed_splits['strict_val'] = strict_val.reset_index(drop=True)
  processed_splits['strict_test'] = strict_test.reset_index(drop=True)

  print(f"val        : {len(processed_splits['val']):,}")
  print(f"strict_val : {len(processed_splits['strict_val']):,}")
  print(f"test       : {len(processed_splits['test']):,}")
  print(f"strict_test: {len(processed_splits['strict_test']):,}")

val        : 971
strict_val : 933
test       : 976
strict_test: 958


## **03. Creating Corpus**

### **3.1. Creating Corpus for Document**

In [34]:
original_split_name = ['train', 'strict_val', 'strict_test']

all_qa = pd.concat([processed_splits[name] for name in original_split_name], ignore_index=True)

doc_corpus = (all_qa[['doc_id', 'document', 'article', 'context', 'content_hash']].drop_duplicates(subset=['doc_id'], keep='first')).reset_index(drop=True)

doc_corpus['embedding_text'] = ("Văn bản: " + doc_corpus['document'] + "\n" + doc_corpus['context'])

doc_corpus['title'] = (doc_corpus['document'] + " - " + doc_corpus['article'])

# Số câu hỏi được đặt ra cho mỗi passage
qa_count = (all_qa.groupby('doc_id').size().rename('qa_count'))

doc_corpus = doc_corpus.join(qa_count, on= 'doc_id', how= 'left')

print(f"Số unique document  trong corpus: {len(doc_corpus):,}")

assert doc_corpus['doc_id'].is_unique
assert doc_corpus['content_hash'].is_unique, "Còn passage trùng nội dung nhưng khác doc_id."

display(doc_corpus.head(3))

Số unique document  trong corpus: 294


,doc_id,document,article,context,content_hash,embedding_text,title,qa_count
0,DOC_40231d053f0f29e1bb80,QUY ĐỊNH ĐÀO TẠO CHƯƠNG TRÌNH TÀI NĂNG,Điều 9. Tuyển bổ sung và loại ra khỏi chương t...,Điều 9.\tTuyển bổ sung và loại ra khỏi chương...,CONTENT_7fce8ab9b5ca93015231,Văn bản: QUY ĐỊNH ĐÀO TẠO CHƯƠNG TRÌNH TÀI NĂN...,QUY ĐỊNH ĐÀO TẠO CHƯƠNG TRÌNH TÀI NĂNG - Điều ...,70
1,DOC_91760158c35d7a2cf252,QUY ĐỊNH ĐÀO TẠO NGOẠI NGỮ ĐỐI VỚI HỆ ĐẠI HỌC ...,Điều 4. Kiểm tra xếp lớp đầu khóa cho sinh viê...,Điều 4. Kiểm tra xếp lớp đầu khóa cho ...,CONTENT_8ce05a028a6c8e694978,Văn bản: QUY ĐỊNH ĐÀO TẠO NGOẠI NGỮ ĐỐI VỚI HỆ...,QUY ĐỊNH ĐÀO TẠO NGOẠI NGỮ ĐỐI VỚI HỆ ĐẠI HỌC ...,68
2,DOC_94f640511e3671b2f2a1,QUY ĐỊNH ĐÀO TẠO CHƯƠNG TRÌNH CHẤT LƯỢNG CAO,Điều 5. Chương trình đào tạo,Điều 5. Chương trình đào tạo\nCT CLC đ...,CONTENT_2b671f3959191064a689,Văn bản: QUY ĐỊNH ĐÀO TẠO CHƯƠNG TRÌNH CHẤT LƯ...,QUY ĐỊNH ĐÀO TẠO CHƯƠNG TRÌNH CHẤT LƯỢNG CAO -...,45


### **3.2. Creating Corpus for QA pairs**

In [35]:
qa_corpus = all_qa.drop_duplicates(subset=['qa_id'], keep='first').reset_index(drop=True)

# Kết thêm các thông tin về content_hash từ doc_corpus.
qa_corpus = pd.merge(
    qa_corpus.drop(columns = ['content_hash']),
    doc_corpus[['doc_id', 'title', 'embedding_text', 'content_hash']].astype({'doc_id': str}),
    on='doc_id',
    how='left'
)

qa_corpus['positive_doc_id'] = qa_corpus['doc_id']

assert qa_corpus['title'].notna().all(), "Có qa_pair không tìm thấy passage tương ứng."

print(f"Số QA_Pair unique: {len(qa_corpus):,}")
display(qa_corpus.head(3))

Số QA_Pair unique: 9,609


,index,context,article,document,question,extractive answer,abstractive answer,yes/no,context_key,document_raw,...,doc_id,query,query_id,qa_id,split,source_index,title,embedding_text,content_hash,positive_doc_id
0,8276,Điều 9.\tTuyển bổ sung và loại ra khỏi chương...,Điều 9. Tuyển bổ sung và loại ra khỏi chương t...,QUY ĐỊNH ĐÀO TẠO CHƯƠNG TRÌNH TÀI NĂNG,Sinh viên dự bị không trở thành sinh viên chín...,02 học kỳ liên tiếp,Sinh viên dự bị không trở thành sinh viên chín...,NaN,điều 9. tuyển bổ sung và loại ra khỏi chương t...,QUY ĐỊNH ĐÀO TẠO CHƯƠNG TRÌNH TÀI NĂNG,...,DOC_40231d053f0f29e1bb80,Sinh viên dự bị không trở thành sinh viên chín...,QUERY_12fc5b07d17e8d48e68a,QA_793b5ec225c56f6fd986,train,8276,QUY ĐỊNH ĐÀO TẠO CHƯƠNG TRÌNH TÀI NĂNG - Điều ...,Văn bản: QUY ĐỊNH ĐÀO TẠO CHƯƠNG TRÌNH TÀI NĂN...,CONTENT_7fce8ab9b5ca93015231,DOC_40231d053f0f29e1bb80
1,8543,Điều 4. Kiểm tra xếp lớp đầu khóa cho ...,Điều 4. Kiểm tra xếp lớp đầu khóa cho sinh viê...,QUY ĐỊNH ĐÀO TẠO NGOẠI NGỮ ĐỐI VỚI HỆ ĐẠI HỌC ...,Các mức điểm xếp lớp tiếng Anh được Hiệu trưởn...,theo đề xuất hợp lý của bộ phận chuyên môn,Các mức điểm xếp lớp tiếng Anh được Hiệu trưởn...,NaN,điều 4. kiểm tra xếp lớp đầu khóa cho sinh viê...,QUY ĐỊNH ĐÀO TẠO NGOẠI NGỮ ĐỐI VỚI HỆ ĐẠI HỌC ...,...,DOC_91760158c35d7a2cf252,Các mức điểm xếp lớp tiếng Anh được Hiệu trưởn...,QUERY_4dae85601229b84f101e,QA_8e639ffaf6e19970c53d,train,8543,QUY ĐỊNH ĐÀO TẠO NGOẠI NGỮ ĐỐI VỚI HỆ ĐẠI HỌC ...,Văn bản: QUY ĐỊNH ĐÀO TẠO NGOẠI NGỮ ĐỐI VỚI HỆ...,CONTENT_8ce05a028a6c8e694978,DOC_91760158c35d7a2cf252
2,7152,Điều 5. Chương trình đào tạo\nCT CLC đ...,Điều 5. Chương trình đào tạo,QUY ĐỊNH ĐÀO TẠO CHƯƠNG TRÌNH CHẤT LƯỢNG CAO,Trình độ tiếng Nhật đạt N mấy mới thì sinh viê...,N3,Sinh viên CT CLC phải đạt tiếng Nhật N3 thì mớ...,NaN,điều 5. chương trình đào tạo ct clc được xây d...,QUY ĐỊNH ĐÀO TẠO CHƯƠNG TRÌNH CHẤT LƯỢNG CAO,...,DOC_94f640511e3671b2f2a1,Trình độ tiếng Nhật đạt N mấy mới thì sinh viê...,QUERY_697317881c5dc7ef7783,QA_49bc47931326cc0e9033,train,7152,QUY ĐỊNH ĐÀO TẠO CHƯƠNG TRÌNH CHẤT LƯỢNG CAO -...,Văn bản: QUY ĐỊNH ĐÀO TẠO CHƯƠNG TRÌNH CHẤT LƯ...,CONTENT_2b671f3959191064a689,DOC_94f640511e3671b2f2a1


In [36]:
qa_corpus.columns

Index(['index', 'context', 'article', 'document', 'question',
       'extractive answer', 'abstractive answer', 'yes/no', 'context_key',
       'document_raw', 'article_raw', 'doc_id', 'query', 'query_id', 'qa_id',
       'split', 'source_index', 'title', 'embedding_text', 'content_hash',
       'positive_doc_id'],
      dtype='object')

## **04. Tokenization**

In [37]:
from transformers import AutoTokenizer

TOKENZIER_MODEL_NAME = "BAAI/bge-m3"

tokenizer = AutoTokenizer.from_pretrained(TOKENZIER_MODEL_NAME, trust_remote_code= True)

def token_length(text: str) -> int:
  return len(tokenizer.encode(text, add_special_tokens= True, truncation= False))

token_stats_source = all_qa[['question', 'context']].drop_duplicates().copy()

token_stats_source['question_tokens'] = (token_stats_source['question'].map(token_length))

token_stats_source['context_tokens'] = (token_stats_source['context'].map(token_length))

doc_corpus['embedding_text_tokens'] = (doc_corpus['embedding_text'].map(token_length))

token_summary = pd.DataFrame({
    'question_tokens': token_stats_source['question_tokens'].describe(percentiles= [0.5, 0.85, 0.9, 0.95, 0.99]),
    'context_tokens': token_stats_source['context_tokens'].describe(percentiles= [0.5, 0.85, 0.9, 0.95, 0.99]),
    'embedding_text_tokens': doc_corpus['embedding_text_tokens'].describe(percentiles= [0.5, 0.85,  0.9, 0.95, 0.99])
})

display(token_summary)

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

,question_tokens,context_tokens,embedding_text_tokens
count,9609.000000,9609.000000,294.000000
mean,21.682069,446.817983,313.948980
std,8.122063,321.418538,247.071836
min,6.000000,21.000000,48.000000
50%,20.000000,348.000000,238.000000
85%,29.000000,774.000000,535.100000
90%,32.000000,1013.000000,607.100000
95%,37.000000,1107.000000,826.750000
99%,48.000000,1324.000000,1137.150000
max,81.000000,1516.000000,1557.000000


In [38]:
token_summary.to_csv(PROCESSED_DIR / 'token_length_sumary.csv', encoding= 'utf-8-sig')

token_stats_source.to_parquet(PROCESSED_DIR / 'token_length_stats.parquet', index = False)

In [39]:
# Tỷ lệ passage bị cắt ứng với từng ngưỡng max_length phổ biến.
truncation_report = []

for max_length in [256, 512, 768, 1024, 2048]:
  truncated_ratio = (doc_corpus['embedding_text_tokens'] > max_length).mean()

  truncation_report.append({
      'max_passage_length': max_length,
      'truncated_passages': int((doc_corpus['embedding_text_tokens'] > max_length).sum()),
      'truncated_ratio': round(float(truncated_ratio), 4),
  })

display(pd.DataFrame(truncation_report))

,max_passage_length,truncated_passages,truncated_ratio
0,256,132,0.4490
1,512,51,0.1735
2,768,19,0.0646
3,1024,10,0.0340
4,2048,0,0.0000


`max_question_length`: có thể chọn gần percentile 99 của câu hỏi

`max_passage_length`: có thể gần 95 hoặc 99 của embeding_text

Nếu nhiều passage vượt quá 512 (số thông thường) thì có thể tăng lên 768 hoặc 1024

- `max_query_length`: 64
- `max_passage_length`: 768

## **05. Export data**

### **5.1. Xuất dữ liệu corpus và master table**

#### **5.1.1. Corpus**

In [40]:
corpus_export_columns = [
    'doc_id',
    'title',
    'document',
    'article',
    'context',
    'qa_count',
    'content_hash',
    'embedding_text']

CORPUS_DIR = Path(PROCESSED_DIR / 'corpus')
CORPUS_DIR.mkdir(exist_ok= True)

In [41]:
corpus_export = doc_corpus[corpus_export_columns].copy()

corpus_export.to_parquet(CORPUS_DIR / 'corpus.parquet', index= False)

dataframe_to_jsonl(corpus_export, CORPUS_DIR / "corpus.jsonl"
)

#### **5.1.2. Queries**

In [42]:
queries_export_columns = [
    'query_id',
    'query',
    'split'
]

In [43]:
query_export = (qa_corpus[queries_export_columns].drop_duplicates(subset=['query_id'], keep='first').reset_index(drop=True))

query_export.to_parquet(CORPUS_DIR / 'queries.parquet', index= False)

dataframe_to_jsonl(query_export, CORPUS_DIR / "queries.jsonl"
)

In [45]:
for split_name in ['train', 'val', 'test']:
  split_query_export = query_export[query_export['split'] == split_name].copy()

  split_query_export.to_parquet(CORPUS_DIR / f'{split_name}_queries.parquet', index=False)

  dataframe_to_jsonl(split_query_export, CORPUS_DIR / f"{split_name}_queries.jsonl")

  print(f"{split_name}_queries: {len(split_query_export):,}")

train_queries: 7,697
val_queries: 933
test_queries: 958


#### **5.1.3. Answer**

In [48]:
answer_export_column = [
    'qa_id',
    'query_id',
    'doc_id',
    'extractive answer',
    'abstractive answer',
    'yes/no',
    'split'
]

In [51]:
answer_export = qa_corpus[answer_export_column].copy()

# Convert relevant columns to string type to avoid ArrowTypeError
answer_export['extractive answer'] = answer_export['extractive answer'].astype(str)
answer_export['abstractive answer'] = answer_export['abstractive answer'].astype(str)
answer_export['yes/no'] = answer_export['yes/no'].astype(str)

answer_export.to_parquet(CORPUS_DIR / 'answer.parquet', index= False)

dataframe_to_jsonl(answer_export, CORPUS_DIR / "answer.jsonl"
)

In [52]:
for split_name in ['train', 'val', 'test']:
  split_answer_export = answer_export[answer_export['split'] == split_name].copy()

  split_answer_export.to_parquet(CORPUS_DIR / f'{split_name}_answer.parquet', index=False)

  dataframe_to_jsonl(split_answer_export, CORPUS_DIR / f"{split_name}_answer.jsonl")

  print(f"{split_name}_answer: {len(split_answer_export):,}")

train_answer: 7,718
val_answer: 933
test_answer: 958


#### **5.1.4. qrels**

In [53]:
qrels_export_columns = [
    'query_id',
    'doc_id',
    'split'
]

In [56]:
qrels_export = qa_corpus[qrels_export_columns].drop_duplicates().reset_index(drop=True)

qrels_export['relevance'] = 1

qrels_export.to_parquet(CORPUS_DIR / 'qrels.parquet', index= False)

dataframe_to_jsonl(qrels_export, CORPUS_DIR / "qrels.jsonl"
)

In [57]:
positive_per_query = qrels_export.groupby('query_id').size()

print(f"Đã xuất qrels: {len(qrels_export):,} cặp (query, doc).")
print(f"Số query có nhiều hơn 1 positive: {int(positive_per_query.gt(1).sum()):,}")

Đã xuất qrels: 9,609 cặp (query, doc).
Số query có nhiều hơn 1 positive: 20


In [58]:
for split_name in ['train', 'val', 'test']:
  split_qrels_export = qrels_export[qrels_export['split'] == split_name].copy()

  split_qrels_export.to_parquet(CORPUS_DIR / f'{split_name}_qrels.parquet', index=False)

  dataframe_to_jsonl(split_qrels_export, CORPUS_DIR / f"{split_name}_qrels.jsonl")

  print(f"{split_name}_qrels: {len(split_qrels_export):,}")

train_qrels: 7,718
val_qrels: 933
test_qrels: 958


#### **5.1.5. Master table**

In [ ]:
qa_corpus.columns

Index(['index', 'context', 'article', 'document', 'question',
       'extractive answer', 'abstractive answer', 'yes/no', 'doc_id', 'query',
       'query_id', 'qa_id', 'split', 'source_index', 'title', 'embedding_text',
       'content_hash', 'positive_doc_id'],
      dtype='object')

In [ ]:
doc_corpus.columns

Index(['doc_id', 'document', 'article', 'context', 'embedding_text', 'title',
       'qa_count', 'content_hash', 'embedding_text_tokens'],
      dtype='object')

In [59]:
master_columns = [
    "qa_id",
    'query_id',
    "doc_id",
    "positive_doc_id",
    "question",
    "query",
    "split",
    "source_index",
    "document",
    "article",
    "context",
    'content_hash',
    'title',
    'embedding_text',
    "extractive answer",
    "abstractive answer",
    "yes/no",
]

In [61]:
qa_corpus['extractive answer'] = qa_corpus['extractive answer'].astype(str)
qa_corpus['abstractive answer'] = qa_corpus['abstractive answer'].astype(str)
qa_corpus['yes/no'] = qa_corpus['yes/no'].astype(str)

qa_corpus[master_columns].to_parquet(CORPUS_DIR / 'qa_master.parquet', index= False)

print("Đã xuất corpus và QA master.")

Đã xuất corpus và QA master.


### **5.1.6. QA_Pair**

In [62]:
qa_pair_columns = [
    "qa_id",
    'query_id',
    "doc_id",
    "question",
    "query",
    "split",
    "extractive answer",
    "abstractive answer",
    "yes/no",
]

In [63]:
qa_pair_export = qa_corpus[qa_pair_columns].copy()

qa_pair_export.to_parquet(CORPUS_DIR / 'qa_pair.parquet', index= False)

dataframe_to_jsonl(qa_pair_export, CORPUS_DIR / "qa_pair.jsonl"
)

print("Đã xuất qa_pair.")

Đã xuất qa_pair.


In [64]:
final_checks = {
    'corpus_size': len(corpus_export),
    'corpus_doc_id_unique': bool(corpus_export['doc_id'].is_unique),
    'corpus_content_unique': bool(corpus_export['content_hash'].is_unique),
    'queries_total': len(query_export),
    'queries_query_id_unique': bool(query_export['query_id'].is_unique),
    'qrels_total': len(qrels_export),
    'qrels_doc_id_in_corpus': bool(qrels_export['doc_id'].isin(set(corpus_export['doc_id'])).all()),
    'qrels_query_id_in_queries': bool(qrels_export['query_id'].isin(set(query_export['query_id'])).all()),
    'train_val_query_leak': len(
        set(query_export.loc[query_export['split'] == 'train', 'query_id'])
        & set(query_export.loc[query_export['split'] == 'val', 'query_id'])
    ),
    'train_test_query_leak': len(
        set(query_export.loc[query_export['split'] == 'train', 'query_id'])
        & set(query_export.loc[query_export['split'] == 'test', 'query_id'])
    ),
    'val_test_query_leak': len(
        set(query_export.loc[query_export['split'] == 'val', 'query_id'])
        & set(query_export.loc[query_export['split'] == 'test', 'query_id'])
    ),
}

for check_name, check_value in final_checks.items():
  print(f"{check_name:28s}: {check_value}")

assert final_checks['corpus_doc_id_unique']
assert final_checks['corpus_content_unique']
assert final_checks['queries_query_id_unique']
assert final_checks['qrels_doc_id_in_corpus']
assert final_checks['qrels_query_id_in_queries']
assert final_checks['train_val_query_leak'] == 0
assert final_checks['train_test_query_leak'] == 0
assert final_checks['val_test_query_leak'] == 0

save_json(final_checks, PROCESSED_DIR / 'preprocessing_checks.json')

print("\nTất cả kiểm tra đã pass.")

corpus_size                 : 294
corpus_doc_id_unique        : True
corpus_content_unique       : True
queries_total               : 9588
queries_query_id_unique     : True
qrels_total                 : 9609
qrels_doc_id_in_corpus      : True
qrels_query_id_in_queries   : True
train_val_query_leak        : 0
train_test_query_leak       : 0
val_test_query_leak         : 0

Tất cả kiểm tra đã pass.
